# Sprint 4 — Serving the Model with FastAPI

## Objective

Build a FastAPI prediction service that loads the serialized cardiovascular disease model and its preprocessing objects.

The API will:

- Load the trained model and saved scaler.
- Validate incoming patient data using Pydantic.
- Apply the same preprocessing used during training.
- Generate a cardiovascular disease prediction.
- Return the prediction as a JSON response.
- Provide an interactive `/docs` interface for testing.

## Prediction Flow

Request → Pydantic Validation → Preprocessing → Model → Prediction → JSON Response

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import tensorflow as tf

In [3]:

model = tf.keras.models.load_model("final_neural_network.keras")
scaler = joblib.load("standard_scaler.joblib")

print("Model loaded successfully.")
print("Scaler loaded successfully.")

Model loaded successfully.
Scaler loaded successfully.


In [4]:
print("Model input shape:", model.input_shape)
print("Scaler expects features:", scaler.n_features_in_)

Model input shape: (None, 14)
Scaler expects features: 14


In [5]:
class PatientData(BaseModel):
    gender: float
    height: float
    weight: float
    ap_hi: float
    ap_lo: float
    cholesterol: float
    gluc: float
    smoke: float
    alco: float
    active: float
    age_years: float
    bmi: float
    pulse_pressure: float
    map: float

In [6]:
app = FastAPI(
    title="Cardiovascular Disease Prediction API",
    description="API for predicting cardiovascular disease using a trained neural network.",
    version="1.0.0"
)

print("FastAPI app created successfully.")

FastAPI app created successfully.


In [7]:
@app.post("/predict")
def predict(data: PatientData):
    # Arrange features in the same order used during training
    features = np.array([[
        data.gender,
        data.height,
        data.weight,
        data.ap_hi,
        data.ap_lo,
        data.cholesterol,
        data.gluc,
        data.smoke,
        data.alco,
        data.active,
        data.age_years,
        data.bmi,
        data.pulse_pressure,
        data.map
    ]])

    # Apply the saved preprocessing
    scaled_features = scaler.transform(features)

    # Get prediction probability
    probability = float(model.predict(scaled_features, verbose=0)[0][0])

    # Apply the optimized threshold
    threshold = 0.42
    prediction = int(probability >= threshold)

    return {
        "prediction": prediction,
        "probability": probability,
        "threshold": threshold
    }

In [8]:
for route in app.routes:
    print(route.path, route.methods)

/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/predict {'POST'}


In [16]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


INFO:     Started server process [985]


In [22]:
from fastapi.responses import RedirectResponse

@app.get("/")
def root():
    return RedirectResponse(url="/docs")

In [23]:
from google.colab import output

output.serve_kernel_port_as_iframe(8000)

<IPython.core.display.Javascript object>

# Day 2 — FastAPI Prediction API

## Completed Tasks

* Built a FastAPI application for the cardiovascular disease prediction model.
* Loaded the serialized neural network model (`final_neural_network.keras`).
* Loaded the saved preprocessing scaler (`standard_scaler.joblib`).
* Defined a Pydantic input schema containing the 14 model features.
* Created a `POST /predict` endpoint.
* Applied the saved scaler before model inference.
* Used the optimized classification threshold of `0.42`.
* Tested the API through FastAPI Swagger documentation (`/docs`).
* Tested multiple valid patient inputs successfully.
* Tested invalid input and confirmed that Pydantic rejects it with HTTP `422`.

## Prediction Flow

Request → Pydantic Validation → Saved Scaler → Neural Network → Probability → Threshold → Prediction

## Test Results

| Test Case     | HTTP Status | Probability | Threshold | Prediction |
| ------------- | ----------- | ----------- | --------- | ---------- |
| Test 1        | 200         | 0.8202      | 0.42      | 1          |
| Test 2        | 200         | 0.4115      | 0.42      | 0          |
| Test 3        | 200         | 0.0470      | 0.42      | 0          |
| Invalid Input | 422         | —           | —         | Rejected   |

## Conclusion

The FastAPI prediction service was successfully implemented and tested.

The API correctly validates patient inputs, applies the saved preprocessing pipeline, generates predictions using the serialized neural network, and returns the prediction probability and classification threshold as a JSON response.
